In [2]:
!nvidia-smi

Sun Apr 12 00:39:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%%writefile matmul.cu
#include<stdio.h>
//matrix multiplication
__global__ void  matMul(float* A, float* B, float* C, int n){
    int row = blockIdx.y*blockDim.y + threadIdx.y;
    int col = blockIdx.x*blockDim.x + threadIdx.x;
    float sum =0.0f;
    for(int i =0; i<1024.0; i++){
        sum += A[row*n+i]*B[i*n + col];  // row start hoti hai from index row*n and then go i steps ahead
    }
    C[row*n+col]=sum;
}
int main(){
  int n = 1024;
  size_t size = n*n*sizeof(float);
  dim3 threadsPerBlock(16,16); // 16x16 = 256 threads per block but in 2d
  dim3 blocksPergrid(n/16,n/16); // sufficient to cover an nxn mtx
  float *ha = (float*)malloc(size);
  float *hb = (float*)malloc(size);
  float *hc = (float*)malloc(size);
  for(int i=0; i< n;i++){
    for(int j =0; j<n;j++){
      ha[i*n+j] =i+j;
      hb[i*n+j]= i*j;
    }
  }
  float *da,*db,*dc;
  cudaMalloc(&da,size);

  cudaMalloc(&db,size);
  cudaMalloc(&dc,size);
  cudaMemcpy(da,ha,size,cudaMemcpyHostToDevice);
  cudaMemcpy(db,hb,size,cudaMemcpyHostToDevice);
  matMul<<<blocksPergrid,threadsPerBlock>>>(da,db,dc,n);
  cudaError_t err = cudaGetLastError();
  if(err != cudaSuccess)
    printf("Kernel error: %s\n", cudaGetErrorString(err));
  cudaDeviceSynchronize();
  cudaMemcpy(hc,dc,size,cudaMemcpyDeviceToHost);
  cudaMemcpy(hc,dc,size,cudaMemcpyDeviceToHost);
  printf("C[0][1] = %.1f\n", hc[0*n + 1]);
  cudaFree(da);cudaFree(db);cudaFree(dc);
  free(ha);free(hb);free(hc);
  return 0;
}

Writing matmul.cu


In [3]:
!nvcc matmul.cu -o matmul && ./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0][1] = 357389440.0
